# FashionSense AI — Fashion Recommendations
An AI stylist: you give your **budget, item types, colours and aesthetic** → the models pick a *coordinated* outfit from the **`arturayupov/womens-fashion-catalog`** Hugging Face dataset (Livostyle women's catalogue, real product images) and show it as product image cards with an estimated total.

**How the two ML models are used**
* **Logistic Regression** (text) scores *every* product for your aesthetic → shortlist per item type.
* **CNN** (product image) re-scores the shortlisted photos **and** produces a visual embedding; pieces with similar embeddings look good *together*, which drives the outfit-coherence score.

**Notes**
* This dataset is *not* H&M — it is the Livostyle catalogue, so no H&M logo is shown by default. To show your own (authorised) brand logo, upload it as `logo.png` (or .svg/.jpg).
* Prices in the dataset are USD; they are converted to ₹ with `USD_TO_INR` (editable, approximate).
* The notebook auto-detects the dataset's columns and prints them. If a detection is wrong, set it in the `COLS` override in section 2.
* Run all cells (Runtime ▸ Run all). The website UI is the last cell.

## 1. Setup

In [ ]:
!pip -q install datasets ipywidgets pillow scikit-learn pyarrow requests

In [ ]:
import os, re, io, glob, base64, random, itertools, warnings
import numpy as np, pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
warnings.filterwarnings("ignore")
random.seed(42); np.random.seed(42)

DATASET, CONFIG = "arturayupov/womens-fashion-catalog", "images"
USD_TO_INR = 88.0        # approximate – edit as needed
MAX_PRODUCTS = 3000      # products kept (one image each)
GITHUB_PARQUET = "https://raw.githubusercontent.com/arturayupov/womens-fashion-catalog-open-data/main/data/products.parquet"  # fallback metadata

def inr(n):
    s = str(int(round(n)))
    if len(s) <= 3: return "₹" + s
    return "₹" + re.sub(r"(\d)(?=(\d\d)+$)", r"\1,", s[:-3]) + "," + s[-3:]

## 2. Load the dataset from Hugging Face

In [ ]:
from datasets import load_dataset, Image as HFImage
ds = load_dataset(DATASET, CONFIG)
split = "train" if "train" in ds else list(ds.keys())[0]
raw = ds[split]
print(raw)
print(raw.features)
img_col = next((k for k, v in raw.features.items() if isinstance(v, HFImage)), None)
tab = (raw.remove_columns([img_col]) if img_col else raw).to_pandas()
tab["_row"] = np.arange(len(tab))
if "image_index" in tab.columns: tab = tab.sort_values("image_index")      # first image of each product comes first

# Column auto-detection. If something is wrong, put the correct column name here, e.g. COLS = {"price": "price_usd"}
COLS = {}
CANDS = {"title": ["title","product_title","name","product_name"],
         "category": ["product_type","category_full_path","category","type"],
         "price": ["price_min_usd","price","min_price","price_min","price_max_usd"],
         "key": ["handle","product_handle","product_id","id","sku"],
         "desc": ["description","body_html","body","tags"],
         "colour": ["color","colour","variant_color"],
         "imgurl": ["image_url","featured_image_url","image_src","src"]}
def detect(t):
    low = {c.lower(): c for c in t.columns}
    return {k: COLS.get(k) or next((low[x] for x in v if x in low), None) for k, v in CANDS.items()}
cc = detect(tab)
if cc["title"] is None or cc["price"] is None:               # the image config may hold only images + ids -> merge product metadata
    print("Title/price not in this config – merging product metadata from the open-data parquet ...")
    prod = pd.read_parquet(GITHUB_PARQUET); pc = detect(prod)
    kt, kp = cc["key"], pc["key"]
    if kt is None or kp is None: raise ValueError(f"Cannot join metadata. Columns: {list(tab.columns)} / {list(prod.columns)} – set COLS manually.")
    tab = tab.assign(_k=tab[kt].astype(str).str.lower()).merge(prod.assign(_k=prod[kp].astype(str).str.lower()), on="_k", how="left", suffixes=("", "_p"))
    cc = detect(tab)
print("Detected columns:", cc)
for need in ("title", "price"):
    if cc[need] is None: raise ValueError(f"Could not find a '{need}' column in {list(tab.columns)} – set COLS = {{'{need}': '<column>'}}")
tab = tab.drop_duplicates(cc["key"] or cc["title"]).head(MAX_PRODUCTS).reset_index(drop=True)   # one image row per product

def S_(col): return tab[col].astype(str).fillna("") if col else pd.Series([""] * len(tab))
df = pd.DataFrame({
    "article_id": np.arange(len(tab)), "row": tab["_row"].values, "name": S_(cc["title"]).str.strip(),
    "category_raw": S_(cc["category"]), "desc": S_(cc["desc"]).str.replace(r"<[^>]+>", " ", regex=True),
    "colour_raw": S_(cc["colour"]), "img_url": S_(cc["imgurl"]),
    "price_usd": pd.to_numeric(S_(cc["price"]).str.replace(r"[^0-9.]", "", regex=True), errors="coerce")})
df = df.dropna(subset=["price_usd"]); df = df[df.price_usd > 0]
df["price"] = np.round(df.price_usd * USD_TO_INR / 10) * 10 - 1

ITEM_GROUPS = ["Top", "Trousers", "Skirt / Shorts", "Dress", "Jacket / Coat", "Two-piece set", "Shoes", "Bag", "Accessories"]
ITEMS = ITEM_GROUPS
RULES = [("Two-piece set", r"\b(two[- ]piece|2[- ]piece|matching set|co-?ord|\bset\b)"),
         (None, r"\b(swim|bikini|one-piece swim|swimsuit|cover-?up)"),
         ("Shoes", r"\b(sandal|heel|sneaker|boot|shoe|flats?|loafer|mule|pump|slipper|wedge|platform)"),
         ("Bag", r"\b(bag|tote|clutch|purse|backpack|crossbody)"),
         ("Accessories", r"\b(belt|scarf|sunglass|hat|cap|jewel|necklace|earring|bracelet|ring|hair|headband|glove|watch)"),
         ("Dress", r"\b(dress|jumpsuit|romper|playsuit|gown)"),
         ("Jacket / Coat", r"\b(jacket|coat|blazer|cardigan|shacket|trench|vest|puffer)"),
         ("Skirt / Shorts", r"\b(skirt|skort|shorts?)"),
         ("Trousers", r"\b(pant|jean|trouser|legging|jogger|palazzo|sweatpant|culotte|cargo)"),
         ("Top", r"\b(top|tee|t-shirt|shirt|blouse|tank|bodysuit|sweater|hoodie|cami|tube|crop|corset|sweatshirt|knit|bralette|bustier)")]
def to_slot(row):
    for t in (row["name"].lower(), (row["category_raw"] + " " + row["name"]).lower()):   # title first, then category
        for slot, pat in RULES:
            if re.search(pat, t): return slot if slot else "drop"
    return "drop"
df["slot"] = df.apply(to_slot, axis=1)
df = df[df.slot != "drop"].reset_index(drop=True)
df["product_type"] = df.category_raw.str.split(",").str[0].str.strip().str.title().replace("", np.nan).fillna(df.slot).str.slice(0, 28)

# ---- product images: downloaded from the dataset's image URLs and cached in IMG_DIR (no placeholders)
import requests
from concurrent.futures import ThreadPoolExecutor
IMG_DIR = "images"; os.makedirs(IMG_DIR, exist_ok=True)
IMG_URL = dict(zip(df.article_id, df.img_url))
def load_pil(aid, size=None):
    p = f"{IMG_DIR}/{int(aid)}.jpg"
    if not os.path.exists(p):
        u = IMG_URL.get(aid, "")
        if not str(u).startswith("http"): return None
        try:
            if "cdn.shopify" in u: u += ("&" if "?" in u else "?") + "width=500"
            im = Image.open(io.BytesIO(requests.get(u, timeout=15).content)).convert("RGB"); im.thumbnail((500, 700)); im.save(p, quality=90)
        except Exception: return None
    try: im = Image.open(p).convert("RGB")
    except Exception: return None
    return im.resize(size) if size else im
def prefetch(ids, workers=16):
    ids = list(ids)
    with ThreadPoolExecutor(workers) as ex: return dict(zip(ids, ex.map(lambda a: load_pil(a, (32, 32)) is not None, ids)))

# ---- colour: from the dataset/title if possible, else estimated from the product photo
COLOUR_FAMILY = {"black":"Black","white":"White","ivory":"White","cream":"Beige","beige":"Beige","tan":"Beige","nude":"Beige","taupe":"Beige","khaki":"Beige",
 "camel":"Brown","brown":"Brown","chocolate":"Brown","mocha":"Brown","grey":"Grey","gray":"Grey","charcoal":"Grey","silver":"Grey","blue":"Blue","navy":"Blue",
 "denim":"Blue","red":"Red","burgundy":"Red","wine":"Red","pink":"Pink","blush":"Pink","green":"Green","olive":"Green","sage":"Green","mint":"Green",
 "yellow":"Yellow","mustard":"Yellow","gold":"Yellow","purple":"Purple","lavender":"Purple","lilac":"Purple","orange":"Orange","rust":"Orange"}
PALETTE = {"Black":(25,25,25),"White":(240,240,240),"Beige":(215,195,165),"Grey":(150,150,150),"Brown":(120,85,60),"Blue":(60,100,170),
 "Red":(190,40,40),"Pink":(240,170,190),"Green":(80,140,90),"Yellow":(240,210,80),"Purple":(130,90,180),"Orange":(235,140,50)}
def text_colour(r):
    for src in (r["colour_raw"], r["name"], r["desc"][:300]):
        m = re.search(r"\b(" + "|".join(COLOUR_FAMILY) + r")\b", src.lower())
        if m: return m.group(1)
    return None
def photo_colour(aid):
    im = load_pil(aid, (64, 64))
    if im is None: return None
    px = np.asarray(im, dtype=float)[16:48, 16:48].reshape(-1, 3)
    px = px[(px.max(1) - px.min(1) > 18) | (px.mean(1) < 200)] if len(px) else px      # ignore plain light backdrop
    c = px.mean(0) if len(px) else np.array([200, 200, 200])
    return min(PALETTE, key=lambda k: np.linalg.norm(c - np.array(PALETTE[k]))).lower()
col = df.apply(text_colour, axis=1)
miss = col.isna()
print(f"Estimating colour from the photo for {int(miss.sum())} products (downloading images, ~1-3 min) ...")
prefetch(df.article_id[miss])
col[miss] = [photo_colour(a) for a in df.article_id[miss]]
df["colour"] = col.fillna("neutral").str.title()
df["family"] = col.map(lambda c: COLOUR_FAMILY.get(c, str(c).title()) if c else "Beige")
df["section"] = "Women"
print(df.shape); df[["name", "product_type", "slot", "colour", "price"]].head()

## 3. Aesthetic tagging (weak labels from product text)

In [ ]:
KEYWORDS = {
 "Minimalist": ["minimal","clean","plain","simple","neutral","straight","relaxed","essential","basic","understated","monochrome","sleek","everyday"],
 "Streetwear": ["oversized","graphic","baggy","cargo","print","utility","washed","boxy","logo","distressed","hoodie","street"],
 "Y2K":        ["low-rise","low rise","rhinestone","cropped","crop","glitter","mini","metallic","sparkle","sparkly","butterfly","flared","baby","y2k","bodycon"],
 "Cottagecore":["floral","puff","linen","smocked","ruffle","prairie","gingham","lace","embroidered","romantic","boho","tiered","peasant","festival"],
 "Classic":    ["tailored","wool","button","pleated","structured","collar","formal","tweed","elegant","classic","blazer","satin","trench","office"],
 "Sporty":     ["stretch","sweat","jersey","track","performance","drawstring","mesh","athletic","active","quick-dry","workout","yoga","gym","sport"],
}
AESTHETICS = list(KEYWORDS)
df["text"] = (df.name.fillna("") + " " + df.desc).str.lower()
for a, kws in KEYWORDS.items():
    df[f"kw_{a}"] = df.text.map(lambda t: sum(t.count(k) for k in kws))
S = df[[f"kw_{a}" for a in AESTHETICS]].values
df["has_label"] = S.max(1) > 0
df["aesthetic"] = np.where(df.has_label, np.array(AESTHETICS)[S.argmax(1)], "Unlabelled")
print(df.aesthetic.value_counts())

## 4. Model 1 — Logistic Regression (text → aesthetic)

In [ ]:
lab = df[df.has_label]
assert lab.aesthetic.nunique() >= 2 and len(lab) > 50, f"Too few keyword-labelled products ({len(lab)}); extend KEYWORDS in section 3."
Xtr, Xte, ytr, yte = train_test_split(lab.text, lab.aesthetic, test_size=.2, stratify=lab.aesthetic, random_state=42)
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
logreg = LogisticRegression(max_iter=2000, C=3.0).fit(tfidf.fit_transform(Xtr), ytr)
pred = logreg.predict(tfidf.transform(Xte)); acc_log = accuracy_score(yte, pred)
print(f"Logistic Regression accuracy: {acc_log:.3f}\n"); print(classification_report(yte, pred))
P = logreg.predict_proba(tfidf.transform(df.text))
for i, a in enumerate(logreg.classes_): df[f"p_{a}"] = P[:, i]    # aesthetic score of EVERY product

## 5. Product images
Images are downloaded from the dataset's image URLs (`load_pil` above) and cached. **No placeholders are generated.** A product whose image fails to load is skipped; if a whole category has no usable image, its card says *Image unavailable*.

In [ ]:
check_images = prefetch
load_pil(df.article_id.iloc[0], (200, 260))

## 6. Model 2 — CNN (product photo → aesthetic + visual embedding)

In [ ]:
CNN_OK, feat_model, acc_cnn = False, None, None
try:
    import tensorflow as tf
    from tensorflow.keras import layers, models
    cand = df[df.has_label].sample(min(800, int(df.has_label.sum())), random_state=0)
    print(f"Loading {len(cand)} product images for the CNN ...")
    ok = check_images(cand.article_id)
    sub = cand[[ok[a] for a in cand.article_id]]
    if len(sub) < 60: raise ImportError(f"only {len(sub)} images available – CNN needs real images")
    Ximg = np.stack([np.asarray(load_pil(a, (64, 64)), dtype="float32") / 255 for a in sub.article_id])
    yidx = sub.aesthetic.map({a: i for i, a in enumerate(AESTHETICS)}).values
    Xa, Xb, ya, yb = train_test_split(Ximg, yidx, test_size=.2, random_state=42)
    inp = layers.Input((64, 64, 3))
    x = layers.MaxPooling2D()(layers.Conv2D(32, 3, activation="relu")(inp))
    x = layers.MaxPooling2D()(layers.Conv2D(64, 3, activation="relu")(x))
    x = layers.GlobalAveragePooling2D()(layers.Conv2D(128, 3, activation="relu")(x))
    emb = layers.Dense(64, activation="relu", name="embedding")(layers.Dropout(.3)(x))
    outp = layers.Dense(len(AESTHETICS), activation="softmax")(emb)
    cnn = models.Model(inp, outp); feat_model = models.Model(inp, [emb, outp])
    cnn.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
    cnn.fit(Xa, ya, validation_data=(Xb, yb), epochs=15, batch_size=32, verbose=2)
    acc_cnn = cnn.evaluate(Xb, yb, verbose=0)[1]; CNN_OK = True
    print(f"\nCNN accuracy: {acc_cnn:.3f}")
except ImportError as e:
    print("CNN skipped:", e)

## 7. Model comparison

In [ ]:
res = {"Logistic Regression": acc_log}
if CNN_OK: res["CNN"] = acc_cnn
plt.figure(figsize=(4, 3)); plt.bar(res.keys(), res.values(), color=["#222", "#999"][:len(res)])
plt.ylim(0, 1); plt.ylabel("Test accuracy"); plt.title("Aesthetic classifiers"); plt.show()

## 8. Recommendation engine (the models drive the outfit)
1. **Filter** by your colours and group by chosen item types.
2. **Logistic Regression** ranks every product per item type for your aesthetic → shortlist (+ a few cheaper options so your budget can be met).
3. Shortlisted photos are loaded; products without a usable image are dropped.
4. **CNN** re-scores each photo for your aesthetic (blended 50/50 with the text score) and gives a visual embedding.
5. Every combination within budget is scored: *aesthetic fit* + *colour harmony* (neutral base, ≤2 accents) + *CNN visual similarity between the pieces*. The best, non-overlapping outfits are returned.

In [ ]:
NEUTRALS = {"Black", "White", "Beige", "Grey", "Brown"}

def recommend(budget, groups, colours, aesthetic, n_looks=3):
    pool = df
    wanted = set()
    for c in colours: wanted |= NEUTRALS if c == "Neutrals" else {c}
    if wanted: pool = pool[pool.family.isin(wanted)]
    pcol = f"p_{aesthetic}"
    k = int(min(6, max(3, round(4000 ** (1 / len(groups))))))
    cand_lists = []
    for g in groups:
        p = pool[pool.slot == g].sort_values(pcol, ascending=False)
        if p.empty: return [], f"No '{g}' products match your colours. Add more colours or another item type."
        short = pd.concat([p.head(12), p.head(30).sort_values("price").head(3)]).drop_duplicates("article_id")
        okimg = check_images(short.article_id)
        short = short.assign(has_img=short.article_id.map(okimg))
        use = short[short.has_img] if short.has_img.any() else short          # skip products without images
        top = use.sort_values(pcol, ascending=False).head(k)
        c = pd.concat([top, use.sort_values("price").head(2)]).drop_duplicates("article_id").copy()
        c["p_text"] = c[pcol]; c["p_cnn"] = np.nan
        cand_lists.append(c)
    emb = {}
    if CNN_OK:
        allc = pd.concat(cand_lists); allc = allc[allc.has_img]
        if len(allc):
            X = np.stack([np.asarray(load_pil(a, (64, 64)), dtype="float32") / 255 for a in allc.article_id])
            E, Pr = feat_model.predict(X, verbose=0)
            En = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-9)
            ai = AESTHETICS.index(aesthetic)
            for a, e, pr in zip(allc.article_id, En, Pr[:, ai]): emb[a] = (e, float(pr))
    recs = []
    for c in cand_lists:
        r = c.to_dict("records")
        for x in r:
            if x["article_id"] in emb: x["p_cnn"] = emb[x["article_id"]][1]
            x["fit"] = x["p_text"] if np.isnan(x["p_cnn"]) else .5 * x["p_text"] + .5 * x["p_cnn"]
        recs.append(r)
    scored = []
    for combo in itertools.product(*recs):
        total = sum(x["price"] for x in combo)
        if total > budget: continue
        accents = len({x["family"] for x in combo} - NEUTRALS)
        colour_h = .15 * sum(x["family"] in NEUTRALS for x in combo) / len(combo) - .2 * max(0, accents - 2)
        es = [emb[x["article_id"]][0] for x in combo if x["article_id"] in emb]
        vis = float(np.mean([a @ b for a, b in itertools.combinations(es, 2)])) if len(es) > 1 else 0.0
        fit = float(np.mean([x["fit"] for x in combo]))
        scored.append((fit + colour_h + .2 * vis, total, vis, combo))
    looks, used = [], set()
    for sc, total, vis, combo in sorted(scored, key=lambda t: -t[0]):
        ids = {x["article_id"] for x in combo}
        if ids & used: continue
        looks.append(dict(combo=list(combo), total=total, visual=vis)); used |= ids
        if len(looks) == n_looks: break
    if not looks:
        cheapest = sum(min(x["price"] for x in r) for r in recs)
        return [], f"No complete outfit fits {inr(budget)}. The cheapest matching outfit is about {inr(cheapest)} — try raising the budget."
    return looks, ""

## 9. Website rendering (product cards only — no cart, billing or checkout)

In [ ]:
CSS = """<style>
.fs{font-family:'Helvetica Neue',Helvetica,Arial,sans-serif;background:#fff;color:#111;max-width:1040px;margin:0 auto;padding:8px 4px}
.fs-head{display:flex;align-items:center;gap:18px;padding:18px 0;border-bottom:1px solid #e5e5e5;margin-bottom:22px}
.fs-head img{height:38px;width:auto}
.fs-title{font-size:15px;letter-spacing:.22em;text-transform:uppercase;font-weight:500;border-left:1px solid #ccc;padding-left:18px}
.fs h2{font-size:13px;letter-spacing:.2em;text-transform:uppercase;font-weight:600;margin:6px 0 4px}
.fs .sub{color:#777;font-size:13px;margin-bottom:18px}
.fs-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:22px}
@media(max-width:820px){.fs-grid{grid-template-columns:repeat(2,1fr)}}
@media(max-width:520px){.fs-grid{grid-template-columns:1fr}}
.card{background:#fff}
.card .im{width:100%;aspect-ratio:3/4;background:#f6f6f6;display:flex;align-items:center;justify-content:center;overflow:hidden}
.card .im img{width:100%;height:100%;object-fit:cover}
.card .na{color:#999;font-size:12px;letter-spacing:.1em;text-transform:uppercase}
.card .role{font-size:11px;letter-spacing:.16em;text-transform:uppercase;color:#888;margin:10px 0 3px}
.card .nm{font-size:15px;font-weight:600}
.card .mt{font-size:13px;color:#555;margin-top:2px}
.card .pr{font-size:15px;margin-top:6px}
.why{margin:30px 0 10px;padding:18px 20px;background:#f7f7f7;font-size:14px;line-height:1.65}
.why b{letter-spacing:.14em;text-transform:uppercase;font-size:12px;display:block;margin-bottom:6px}
.total{margin-top:26px;padding-top:18px;border-top:1px solid #111;font-size:17px;font-weight:600;letter-spacing:.04em}
</style>"""

def logo_img_tag():
    f = (glob.glob("logo.*") or [None])[0]
    if not f: return ""
    mime = {"svg": "image/svg+xml", "jpg": "image/jpeg", "jpeg": "image/jpeg"}.get(f.rsplit(".", 1)[-1].lower(), "image/png")
    return f'<img alt="logo" src="data:{mime};base64,{base64.b64encode(open(f, "rb").read()).decode()}">'

def header_html():
    return f'<div class="fs-head">{logo_img_tag()}<span class="fs-title">Fashion Recommendations</span></div>'

def card_html(x):
    im = load_pil(x["article_id"])
    if im is not None:
        im.thumbnail((480, 640)); buf = io.BytesIO(); im.save(buf, "JPEG", quality=85)
        body = f'<img alt="{x["name"]}" src="data:image/jpeg;base64,{base64.b64encode(buf.getvalue()).decode()}">'
    else:
        body = '<span class="na">Image unavailable</span>'
    return (f'<div class="card"><div class="im">{body}</div><div class="role">{x["slot"]}</div>'
            f'<div class="nm">{x["name"]}</div><div class="mt">{x["product_type"]} · {x["colour"]}</div>'
            f'<div class="pr">{inr(x["price"])}</div></div>')

NOTES = {"Minimalist": "Clean lines and a restrained palette keep the look effortless and easy to re-wear.",
 "Streetwear": "Relaxed proportions and bold details give the outfit an urban, off-duty attitude.",
 "Y2K": "Cropped, playful details channel the early-2000s mood while the pieces still read as one outfit.",
 "Cottagecore": "Soft textures and romantic details create a gentle, nostalgic look.",
 "Classic": "Structured, timeless pieces make the outfit polished and versatile.",
 "Sporty": "Stretch fabrics and athletic cues keep the outfit comfortable and active."}

def explain(look, aesthetic):
    combo = look["combo"]
    fams = [x["family"] for x in combo]; accents = sorted(set(fams) - NEUTRALS)
    pal = ("a harmonious neutral palette" if not accents else
           f"a neutral base lifted by {' and '.join(a.lower() for a in accents)}")
    txt = (f"This {aesthetic.lower()} outfit pairs " + ", ".join(f"the {x['name'].lower()} ({x['colour'].lower()})" for x in combo[:-1]) +
           (f" with the {combo[-1]['name'].lower()} ({combo[-1]['colour'].lower()})" if len(combo) > 1 else "") + f". The pieces share {pal}. ")
    txt += NOTES[aesthetic] + " "
    t = np.mean([x["p_text"] for x in combo]) * 100
    txt += f"Our text model rates the pieces {t:.0f}% on-aesthetic on average"
    cn = [x["p_cnn"] for x in combo if not np.isnan(x["p_cnn"])]
    if cn: txt += f", the image model gives the product photos a {np.mean(cn)*100:.0f}% match, and their visual similarity is {look['visual']*100:.0f}%, so they look good together."
    else: txt += "."
    return txt

def look_html(look, aesthetic):
    cards = "".join(card_html(x) for x in look["combo"])
    return (CSS + f'<div class="fs">{header_html()}<h2>Your {aesthetic} outfit</h2>'
            f'<div class="sub">Styled by FashionSense AI from the catalogue</div><div class="fs-grid">{cards}</div>'
            f'<div class="why"><b>Why these pieces work together</b>{explain(look, aesthetic)}</div>'
            f'<div class="total">Estimated Total Outfit Price: {inr(look["total"])}</div></div>')

## 10. FashionSense AI — the website
Set your preferences, then press **Generate My Outfit**. The line under the form shows exactly which values were passed to the model.

In [ ]:
import ipywidgets as w
from IPython.display import display, HTML, clear_output
try:
    from google.colab import output; output.enable_custom_widget_manager()
except Exception: pass

COLOUR_OPTIONS = ["Neutrals", "White", "Black", "Beige", "Grey", "Brown", "Blue", "Red", "Pink", "Green", "Yellow", "Purple", "Orange"]
wrap = w.Layout(display="flex", flex_flow="row wrap", gap="6px", width="100%")
def chips(opts, default=()):
    return {o: w.ToggleButton(value=o in default, description=o, layout=w.Layout(width="118px", height="34px")) for o in opts}

budget = w.IntSlider(value=8000, min=1000, max=40000, step=500, readout=False, layout=w.Layout(width="360px"))
budget_box = w.BoundedIntText(value=8000, min=1000, max=40000, step=500, layout=w.Layout(width="110px"))
w.jslink((budget, "value"), (budget_box, "value"))
item_chips = chips(ITEMS, ("Top", "Trousers", "Shoes"))
colour_chips = chips(COLOUR_OPTIONS, ("White", "Beige", "Neutrals"))
aesth = w.Dropdown(options=AESTHETICS, value="Minimalist", layout=w.Layout(width="260px"))
lbl = lambda t: w.HTML(f"<b style='letter-spacing:.14em;text-transform:uppercase;font-size:12px'>{t}</b>")
btn = w.Button(description="Generate My Outfit", button_style="", layout=w.Layout(width="220px", height="42px"))
btn.style.button_color = "#111"; btn.style.text_color = "white"
btn2 = w.Button(description="Show another look", layout=w.Layout(width="180px", height="42px"))
prefs_out, out = w.Output(), w.Output()
state = {"looks": [], "i": 0, "aesthetic": None}

def show():
    with out:
        clear_output(wait=True)
        display(HTML(look_html(state["looks"][state["i"]], state["aesthetic"])))

def generate(_):
    groups = [i for i, b in item_chips.items() if b.value]
    colours = [c for c, b in colour_chips.items() if b.value]
    with prefs_out:
        clear_output(wait=True)
        print(f"Sent to model → budget: {inr(budget.value)} | items: {groups or 'none'} | colours: {colours or 'any'} | aesthetic: {aesth.value}")
    with out:
        clear_output(wait=True)
        if not groups: print("Please select at least one item type."); return
        print("Styling your outfit…")
    looks, msg = recommend(budget.value, groups, colours, aesth.value)
    if not looks:
        with out: clear_output(wait=True); print(msg)
        return
    state.update(looks=looks, i=0, aesthetic=aesth.value); show()

def another(_):
    if state["looks"]: state["i"] = (state["i"] + 1) % len(state["looks"]); show()

btn.on_click(generate); btn2.on_click(another)
display(HTML(CSS + f'<div class="fs">{header_html()}</div>'))
display(w.VBox([
    lbl("Budget (₹)"), w.HBox([budget, budget_box]),
    lbl("Items (choose one or more)"), w.Box(list(item_chips.values()), layout=wrap),
    lbl("Colours (choose one or more)"), w.Box(list(colour_chips.values()), layout=wrap),
    lbl("Aesthetic"), aesth,
    w.HBox([btn, btn2]), prefs_out], layout=w.Layout(width="760px", gap="8px", overflow="visible")))
display(out)